In [27]:
import torch
import torch.nn as nn
import torchmetrics
import torchvision.transforms.v2 as T
import torchvision

from torch.utils.data import DataLoader
import torch.nn.functional as F

import time

In [28]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(dtype=torch.float32, scale=True)])

train_valid_set = torchvision.datasets.CIFAR10(root='datasets', train=True, download=True,transform=toTensor)
test_set = torchvision.datasets.CIFAR10(root='datasets', train=False, download=True,transform=toTensor)

torch.manual_seed(52)
train_set, valid_set = torch.utils.data.random_split(train_valid_set, [45000, 5000])

batch_size = 128
train_data = DataLoader(train_set, batch_size=batch_size)
valid_data = DataLoader(valid_set, batch_size=batch_size)
test_data  = DataLoader(test_set, batch_size=batch_size)

/home/damian/test/test/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [29]:
def he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight.data)
        nn.init.zeros_(module.bias.data)

In [30]:
def build_deep_model(n_hidden, n_neurons, n_inputs, n_outputs):
    layers = [nn.Flatten(), nn.Linear(n_inputs, n_neurons), nn.SiLU()]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(n_neurons, n_neurons), nn.SiLU()]
        
    layers += [nn.Linear(n_neurons, n_outputs)]
    model = nn.Sequential(*layers)
    model.apply(he_init)
    
    return model

In [31]:
def evaluate(model, valid_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        
    return metric.compute()

In [32]:
def train(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs, patience=5, scheduler=None, checkpoint_path=None):
    checkpoint_path = checkpoint_path or "my_checkpoint.pt"
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    best_metric = 0.0
    patience_counter = 0
    for epoch in range(n_epochs):
        total_loss = 0
        metric.reset()
        model.train()
        t0 = time.time()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            
        train_metric = metric.compute().item()
        valid_metric = evaluate(model, valid_loader, metric).item()
            
        if valid_metric > best_metric:
            torch.save(model.state_dict(), checkpoint_path)
            best_metric = valid_metric
            best = " (best)"
            patience_counter = 0
        else: 
            patience_counter += 1
            best = ""
            
        t1 = time.time()
        history["train_losses"].append(total_loss/len(train_loader))
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)
        print(f"epoch: {epoch}, loss: {history['train_losses'][-1]:.4f}, t_score: {(train_metric*100):.2f}%, v_score: {(valid_metric*100):.2f}% {best} in {(t1-t0):.1f}s")
        
        if scheduler is not None:
            scheduler.step()
        if patience_counter > patience:
            print('early stop!')
            break
        
    model.load_state_dict(torch.load(checkpoint_path))
    return history

In [33]:
model_1 = build_deep_model(n_hidden=20, n_neurons=100, n_inputs=3*32*32, n_outputs=10).to('cuda')
optimizer = torch.optim.NAdam(params=model_1.parameters(), lr=0.001)
xentropy = nn.CrossEntropyLoss()

accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
n_epochs=100

In [34]:
history_1 = train(model_1, optimizer, xentropy, train_data, valid_data, accuracy, n_epochs)

epoch: 0, loss: 2.1341, t_score: 18.33%, v_score: 19.44%  (best) in 7.2s
epoch: 1, loss: 1.9276, t_score: 27.23%, v_score: 29.20%  (best) in 7.0s
epoch: 2, loss: 1.8201, t_score: 32.99%, v_score: 33.28%  (best) in 7.1s
epoch: 3, loss: 1.7187, t_score: 37.55%, v_score: 36.14%  (best) in 7.0s
epoch: 4, loss: 1.6582, t_score: 40.34%, v_score: 37.52%  (best) in 7.1s
epoch: 5, loss: 1.6030, t_score: 42.40%, v_score: 37.64%  (best) in 7.0s
epoch: 6, loss: 1.5590, t_score: 44.07%, v_score: 39.62%  (best) in 7.0s
epoch: 7, loss: 1.5232, t_score: 45.62%, v_score: 41.52%  (best) in 6.9s
epoch: 8, loss: 1.5132, t_score: 46.10%, v_score: 39.72%  in 7.0s
epoch: 9, loss: 1.4664, t_score: 47.59%, v_score: 42.82%  (best) in 6.8s
epoch: 10, loss: 1.4386, t_score: 48.45%, v_score: 42.84%  (best) in 7.0s
epoch: 11, loss: 1.4129, t_score: 49.44%, v_score: 43.14%  (best) in 7.0s
epoch: 12, loss: 1.3878, t_score: 50.25%, v_score: 43.60%  (best) in 7.0s
epoch: 13, loss: 1.3702, t_score: 51.00%, v_score: 43.5

In [46]:
def build_deep_model_bn(n_hidden, n_neuron, n_input, n_output):
    layers = [nn.Flatten(), nn.Linear(n_input, n_neuron), nn.BatchNorm1d(n_neuron), nn.SiLU()]
    for _ in range(n_hidden-1):
        layers += [nn.Linear(n_neuron, n_neuron), nn.BatchNorm1d(n_neuron), nn.SiLU()]
    layers += [nn.Linear(n_neuron, n_output)]
    
    model = nn.Sequential(*layers)
    model.apply(he_init)
    return model

In [47]:
model_2 = build_deep_model_bn(20, 100, 3*32*32, 10).to('cuda')

optimizer = torch.optim.NAdam(params=model_2.parameters(), lr=0.001)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')

In [40]:
history_2 = train(model_2, optimizer, xentropy, train_data, valid_data, accuracy, n_epochs)

epoch: 0, loss: 2.0041, t_score: 26.96%, v_score: 36.30%  (best) in 8.0s
epoch: 1, loss: 1.6767, t_score: 39.90%, v_score: 40.78%  (best) in 8.0s
epoch: 2, loss: 1.5371, t_score: 44.81%, v_score: 39.58%  in 7.9s
epoch: 3, loss: 1.4445, t_score: 48.42%, v_score: 44.00%  (best) in 8.1s
epoch: 4, loss: 1.3731, t_score: 51.02%, v_score: 42.88%  in 7.7s
epoch: 5, loss: 1.3114, t_score: 53.50%, v_score: 42.08%  in 8.1s
epoch: 6, loss: 1.2529, t_score: 55.67%, v_score: 41.20%  in 8.2s
epoch: 7, loss: 1.2025, t_score: 57.56%, v_score: 41.18%  in 8.2s
epoch: 8, loss: 1.1589, t_score: 59.08%, v_score: 41.02%  in 7.8s
epoch: 9, loss: 1.1167, t_score: 60.74%, v_score: 42.32%  in 7.6s
early stop!


-------------------------

In [41]:
class standardize(nn.Module):
    def __init__(self, sample):
        super().__init__()
        flat = torch.flatten(sample, start_dim=1)
        mean = flat.mean(dim=0, keepdim=True)
        std  = flat.std(dim=0, keepdim=True)
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        
    def forward(self, x):
        return (x - self.mean) / self.std

In [42]:
all_imgs = torch.stack([img for img, _ in train_set])
standartized = standardize(all_imgs)

In [51]:
def lecun_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='linear')
        nn.init.zeros_(module.bias)

In [52]:
def build_deep_model_selu(n_hidden, n_neuron, n_input, n_output):
    layers = [nn.Flatten(), standartized ,nn.Linear(n_input, n_neuron), nn.SELU()]
    for _ in range(n_hidden-1):
        layers += [nn.Linear(n_neuron, n_neuron), nn.SELU()]
    layers += [nn.Linear(n_neuron, n_output)]
    
    model = nn.Sequential(*layers)
    model.apply(lecun_init)
    return model

In [53]:
model_3 = build_deep_model_selu(20, 100, 3*32*32, 10).to('cuda')

optimizer = torch.optim.SGD(params=model_3.parameters(), lr = 0.001)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')

In [54]:
history_3 = train(model_3, optimizer, xentropy, train_data, valid_data, accuracy, n_epochs)

epoch: 0, loss: 2.0955, t_score: 24.52%, v_score: 29.64%  (best) in 7.0s
epoch: 1, loss: 1.8747, t_score: 32.57%, v_score: 33.68%  (best) in 6.9s
epoch: 2, loss: 1.7915, t_score: 36.00%, v_score: 35.06%  (best) in 6.9s
epoch: 3, loss: 1.7353, t_score: 38.14%, v_score: 36.42%  (best) in 6.9s
epoch: 4, loss: 1.6913, t_score: 39.88%, v_score: 36.88%  (best) in 6.6s
epoch: 5, loss: 1.6557, t_score: 41.12%, v_score: 37.92%  (best) in 6.9s
epoch: 6, loss: 1.6247, t_score: 42.38%, v_score: 38.80%  (best) in 7.0s
epoch: 7, loss: 1.5976, t_score: 43.34%, v_score: 39.34%  (best) in 6.9s
epoch: 8, loss: 1.5730, t_score: 44.28%, v_score: 40.02%  (best) in 6.9s
epoch: 9, loss: 1.5502, t_score: 44.94%, v_score: 40.18%  (best) in 6.8s
epoch: 10, loss: 1.5297, t_score: 45.64%, v_score: 40.70%  (best) in 6.9s
epoch: 11, loss: 1.5101, t_score: 46.27%, v_score: 40.92%  (best) in 6.8s
epoch: 12, loss: 1.4919, t_score: 46.97%, v_score: 40.90%  in 6.8s
epoch: 13, loss: 1.4743, t_score: 47.61%, v_score: 41.2

### 4

In [55]:
def build_deep_model_alphadropout(n_hidden, n_neuron, n_input, n_output, dropout_rate):
    layers = [nn.Flatten(), standartized ,nn.Linear(n_input, n_neuron), nn.SELU(), nn.AlphaDropout(dropout_rate)]
    for _ in range(n_hidden-1):
        layers += [nn.Linear(n_neuron, n_neuron), nn.SELU(), nn.AlphaDropout(dropout_rate)]
    layers += [nn.Linear(n_neuron, n_output)]
    
    model = nn.Sequential(*layers)
    model.apply(lecun_init)
    return model

In [56]:
model_4 = build_deep_model_alphadropout(20, 100, 3*32*32, 10, 0.1).to('cuda')

optimizer = torch.optim.NAdam(params=model_4.parameters(), lr=0.001)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')

In [57]:
train(model_4, optimizer, xentropy, train_data, valid_data, accuracy, n_epochs)

epoch: 0, loss: 2.1593, t_score: 19.08%, v_score: 23.86%  (best) in 7.8s
epoch: 1, loss: 1.9783, t_score: 23.92%, v_score: 27.10%  (best) in 7.7s
epoch: 2, loss: 1.9100, t_score: 26.52%, v_score: 28.16%  (best) in 7.5s
epoch: 3, loss: 1.8598, t_score: 28.42%, v_score: 29.44%  (best) in 7.3s
epoch: 4, loss: 1.8181, t_score: 31.09%, v_score: 31.66%  (best) in 7.6s
epoch: 5, loss: 1.7805, t_score: 32.93%, v_score: 32.62%  (best) in 7.7s
epoch: 6, loss: 1.7445, t_score: 35.29%, v_score: 33.34%  (best) in 7.7s
epoch: 7, loss: 1.7142, t_score: 36.87%, v_score: 36.38%  (best) in 7.4s
epoch: 8, loss: 1.6794, t_score: 38.36%, v_score: 36.78%  (best) in 7.2s
epoch: 9, loss: 1.6654, t_score: 39.64%, v_score: 36.96%  (best) in 7.6s
epoch: 10, loss: 1.6434, t_score: 40.48%, v_score: 38.00%  (best) in 7.6s
epoch: 11, loss: 1.6204, t_score: 41.31%, v_score: 38.26%  (best) in 7.7s
epoch: 12, loss: 1.6416, t_score: 40.92%, v_score: 37.58%  in 7.4s
epoch: 13, loss: 1.6127, t_score: 42.29%, v_score: 37.5

{'train_losses': [2.1593124331398443,
  1.9783487797460773,
  1.9099589498205618,
  1.859821981665763,
  1.818115987222303,
  1.7805005901239135,
  1.744529741731557,
  1.714175501668995,
  1.6793862994421611,
  1.6653683957728473,
  1.6434158042750575,
  1.620447387072173,
  1.641578670591116,
  1.6127397323196584,
  1.5951822830194777,
  1.5724020769650286,
  1.5550663440742276,
  1.5478436144238168,
  1.533500772985545,
  1.52914414385503,
  1.52942127158696,
  1.5089684237133374,
  1.5396558418869972,
  1.5032272647050293,
  1.4791327027434653,
  1.4855473119426856,
  1.4737603904848748,
  1.4641063457185572,
  1.4556946036490528,
  1.4456616701050238,
  1.4454816335981542,
  1.4427834606983445,
  1.4478553015400062,
  1.4325197345831178,
  1.4329160732979125,
  1.447487498548898,
  1.4357788315550848,
  1.4222070910036564,
  1.4154773083600132,
  1.4086855937811462,
  1.4014182426035404,
  1.4181307307021185,
  1.430217088962143,
  1.421893665397709,
  1.4638102816587144],
 'train

# 5

In [60]:
n_epochs=50
model_5 = build_deep_model_alphadropout(20, 100, 3*32*32, 10, 0.1).to('cuda')

optimizer = torch.optim.NAdam(params=model_5.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01, steps_per_epoch=len(train_data), epochs=n_epochs)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')

In [61]:
history_5 = train(model_5, optimizer, xentropy, train_data, valid_data, accuracy, n_epochs, patience=20, scheduler=scheduler)

epoch: 0, loss: 2.2818, t_score: 15.71%, v_score: 18.40%  (best) in 7.7s
epoch: 1, loss: 2.0634, t_score: 20.52%, v_score: 25.18%  (best) in 7.7s
epoch: 2, loss: 1.9648, t_score: 25.31%, v_score: 27.60%  (best) in 7.4s
epoch: 3, loss: 1.9236, t_score: 27.16%, v_score: 26.40%  in 7.6s
epoch: 4, loss: 1.8903, t_score: 28.62%, v_score: 26.34%  in 7.7s
epoch: 5, loss: 1.8601, t_score: 30.51%, v_score: 31.12%  (best) in 7.6s
epoch: 6, loss: 1.8173, t_score: 32.04%, v_score: 32.08%  (best) in 7.6s
epoch: 7, loss: 1.7739, t_score: 33.46%, v_score: 32.20%  (best) in 7.4s
epoch: 8, loss: 1.7417, t_score: 35.27%, v_score: 34.84%  (best) in 7.6s
epoch: 9, loss: 1.7096, t_score: 36.28%, v_score: 36.56%  (best) in 7.6s
epoch: 10, loss: 1.6821, t_score: 37.88%, v_score: 38.12%  (best) in 7.6s
epoch: 11, loss: 1.6527, t_score: 39.12%, v_score: 37.76%  in 7.5s
epoch: 12, loss: 1.6307, t_score: 40.22%, v_score: 38.80%  (best) in 7.3s
epoch: 13, loss: 1.6132, t_score: 41.10%, v_score: 39.24%  (best) in 